# Brazil data validation — INEP Censo + Taxas de Rendimento

**Living Stone Foundation — Applied Data Lab**

**Who this notebook is for:** readers who did **not** build the project.  
**Goal:** prove that the population (Brazilian basic-education **schools**) matches the labels we train on (official INEP **abandonment rates**).


## Variable dictionary (read before the charts)

| Column / label in charts | Plain-English meaning | How to read values |
|---|---|---|
| `target_dropout_rate` / **Abandonment rate (%)** | Official INEP *taxa de abandono*: share of students who **stopped attending** during the school year after the census reference date | 0 = nobody left; 5 = 5% left. Higher = worse |
| `year` | School census / rendimento year | Years present in the mart (currently 2018–2025) |
| `school_id` (`CO_ENTIDADE`) | Unique school code (INEP) | Join key between Censo and Rendimento |
| `uf` | Brazilian state abbreviation | e.g. SP, BA, AM |
| `municipio_id` | IBGE municipality code | Geographic context |
| `tp_dependencia` | Administrative network | 1=Federal, 2=State, 3=Municipal, 4=Private |
| `tp_localizacao` | School location type | 1=Urban, 2=Rural |
| `is_rural` | 1 if rural school | Shortcut of `tp_localizacao == 2` |
| `is_public` | 1 if public network (federal/state/municipal) | 0 = private |
| `in_agua` | Has potable / public water | 1=yes, 0=no |
| `in_energia` | Connected to public electricity | 1=yes, 0=no |
| `in_esgoto` | Public sewage connection | 1=yes, 0=no |
| `in_internet` | Internet available at school | 1=yes, 0=no |
| `in_biblioteca` | Library / reading room | 1=yes, 0=no |
| `in_lab_info` | Computer lab | 1=yes, 0=no |
| `in_quadra` | Sports court | 1=yes, 0=no |
| `qt_mat_bas` | Enrollment count (basic education total, when available) | Larger = bigger school |
| `enrollment_level` | Enrollment used for this level (Fundamental or Médio) | Filter requires ≥ 20 students |
| `qt_doc_bas` | Number of teachers (basic education) | Staffing intensity |
| `student_teacher_ratio` | Students ÷ teachers | Higher often means more crowded classes |
| `risk_band` | low / moderate / high | Relative triage label inside the level |
| `high_risk` | 1 if school is in the elevated-risk group | Binary flag for triage demos |
| **MAE / RMSE / R²** | Model error metrics (later notebooks / model folder) | Lower MAE/RMSE better; R² closer to 1 better |

### Acronyms
| Acronym | Meaning |
|---|---|
| **INEP** | Brazilian federal education statistics institute |
| **Censo Escolar** | Annual school census (structure + enrollment) |
| **Taxas de Rendimento** | Official approval / failure / abandonment rates |
| **EDA** | Exploratory Data Analysis |
| **UF** | Federative unit (state) |
| **Fundamental** | Ensino Fundamental (approx. primary + lower secondary) |
| **Médio** | Ensino Médio (upper secondary) |


### How to read this notebook
1. Run cells top to bottom (`Shift+Enter`).
2. After every code cell, read the **Insight** markdown — that is the takeaway, not the raw JSON.


In [1]:
# Cell A — project path only (no Path.cwd / exists / resolve)
import sys
ROOT = r"C:\Users\User\Desktop\Projeto Living Stone Foundation"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


### What this cell did
Added the project folder to Python’s import path using a **fixed string** (no `Path.cwd()` / `exists` / `resolve`). Those path checks can freeze kernels on Desktop/OneDrive.


In [2]:
# Cell B — imports (first run can take a minute for pandas/seaborn)
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FEATURE_LABELS = {
    "is_rural": "Rural school (1=yes)",
    "is_public": "Public network (1=yes)",
    "in_internet": "Has internet (1=yes)",
    "in_lab_info": "Has computer lab (1=yes)",
    "in_quadra": "Has sports court (1=yes)",
    "in_biblioteca": "Has library (1=yes)",
    "in_agua": "Has water (1=yes)",
    "in_energia": "Has electricity (1=yes)",
    "in_esgoto": "Has sewage (1=yes)",
    "enrollment_level": "Enrollment (this level)",
    "qt_mat_bas": "Basic-ed enrollment (total)",
    "qt_doc_bas": "Number of teachers",
    "student_teacher_ratio": "Students per teacher",
    "tp_dependencia": "Admin network code",
    "tp_localizacao": "Urban/rural code",
    "target_dropout_rate": "Abandonment rate (%)",
}
import json
from src.eda import compare_levels, validation_json_path
print("Imports OK")


Imports OK


### What this cell did
Loaded charting/table libraries. The **first** run can take ~30–90s while pandas/seaborn warm up — that is normal, not a freeze on `import sys`.


## 1. Official validation artifact


In [3]:
path = validation_json_path()
payload = json.loads(path.read_text(encoding='utf-8'))
print('File:', path)
print('Source:', payload.get('source'))
print('Abandonment definition:', payload.get('definition_abandono'))
print('Grain:', payload.get('grain'))
print('Years:', payload.get('years'))
print('Rows / schools:', payload.get('rows'), '/', payload.get('schools'))
print('Coverage:', json.dumps(payload.get('coverage'), indent=2))
print('Student-level labels available?', payload.get('student_level_labeled_dropout_available'))


File: C:\Users\User\Desktop\Projeto Living Stone Foundation\latam_education_data\dq\rendimento_validation.json
Source: INEP Taxas de Rendimento Escolar (official)
Abandonment definition: Percentage of students who stopped attending school after the School Census reference date during the school year (movement = left attending). Computed from Censo Escolar Situação do Aluno module.
Grain: school x year (CO_ENTIDADE / school_id)
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Rows / schools: 1040730 / 147330
Coverage: {
  "fundamental_non_null_pct": 0.925817455055586,
  "medio_non_null_pct": 0.22405907391926821,
  "fundamental_mean": 1.203613394968065,
  "medio_mean": 3.324452258936038
}
Student-level labels available? False


### How to read this output
- **Source** should cite INEP Taxas de Rendimento (official), not a homemade proxy.
- **Definition** means: students who *left attending* during the year — not “failed the year” and not “never enrolled”.
- **Grain = school × year** means each measurement is about a **school**, not a named student.
- **Coverage**
  - `fundamental_non_null_pct` near ~90%+: most schools have a Fundamental abandonment number.
  - `medio_non_null_pct` much lower (~20%+): only schools that offer Ensino Médio report that rate.
  - Means: Médio abandonment is typically **higher** than Fundamental (e.g. ~3% vs ~1%).
- **Student-level labels available? → False**: we cannot honestly build a student scorer with these open files.

### Insight (decision for the project)
Population and labels are aligned for a **Brazil school-level** early-warning system. They are **not** aligned with the old Kaggle higher-education student dataset — that is why it was removed.


## 2. Side-by-side mart summary (Fundamental vs Médio)


In [4]:
summary = compare_levels()
# Friendlier column names for readers
pretty = summary.rename(columns={
    'level': 'Education level',
    'rows': 'School-year rows',
    'schools': 'Unique schools',
    'mean_abandono': 'Mean abandonment (%)',
    'median_abandono': 'Median abandonment (%)',
    'p90_abandono': '90th percentile abandonment (%)',
    'public_share': 'Share public schools',
    'rural_share': 'Share rural schools',
})
display(pretty)


,Education level,School-year rows,Unique schools,Mean abandonment (%),Median abandonment (%),90th percentile abandonment (%),Share public schools,Share rural schools
0,fundamental,832326,122211,1.142,0.000,3.300,0.799,0.313
1,medio,225199,32612,3.314,0.500,10.200,0.722,0.104


### How to read this table
Each row is one education level after joining Censo features to official abandonment rates and keeping schools with enough enrollment (≥ 20).

| Column | Takeaway question |
|---|---|
| School-year rows | Do we have enough data to model? |
| Mean vs median abandonment | Is the outcome rare/skewed? (median near 0 with mean > median ⇒ many zeros + a right tail) |
| 90th percentile | What does a “high” school look like in this level? |
| Public / rural shares | What kind of schools dominate the sample? |

### Insight
If **Médio** shows higher mean abandonment and a heavier right tail than **Fundamental**, training **two models** (and two Streamlit apps) is justified: the risk regimes are different. A single pooled model would mix apples and oranges.
